In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer


model_name = "/code/model_checkpoints/Qwen3-1.7B"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype="auto", device_map="auto"
)

Skipping import of cpp extensions due to incompatible torch version 2.8.0+cu129 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info
`torch_dtype` is deprecated! Use `dtype` instead!


In [1]:
import torch
import torch.nn as nn
import copy
from typing import Optional, Tuple, Union, List

from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForCausalLM,
    PreTrainedModel,
    GenerationMixin,  # Явно импортируем миксин генерации
)
from transformers.modeling_outputs import (
    BaseModelOutputWithPast,
    CausalLMOutputWithPast,
)
from transformers.masking_utils import create_causal_mask

# Проверка наличия библиотеки солвера
try:
    from torchdiffeq import odeint_adjoint as odeint
except ImportError:
    raise ImportError("Пожалуйста, установите библиотеку: pip install torchdiffeq")


# =============================================================================
# 1. БЛОК ДИНАМИКИ (Mixed Precision Wrapper)
# =============================================================================


class ODEDynamicsWrapper(nn.Module):
    def __init__(self, base_layer: nn.Module, hidden_size: int):
        super().__init__()
        self.base_layer = base_layer
        # Time Embedding: позволяет слою знать, в каком "времени" t он находится
        self.time_embed = nn.Sequential(
            nn.Linear(1, hidden_size), nn.SiLU(), nn.Linear(hidden_size, hidden_size)
        )
        self.context = {}

    def set_context(self, **kwargs):
        self.context = kwargs

    def forward(self, t: torch.Tensor, hidden_states: torch.Tensor):
        # 1. Определяем целевой тип данных модели
        target_dtype = self.base_layer.input_layernorm.weight.dtype
        target_device = self.base_layer.input_layernorm.weight.device

        # 2. Кастуем входные данные ВНИЗ (в fp16/bf16) для нейросети
        h_in = hidden_states.to(target_dtype)

        # 3. Обработка времени
        t_val = t.view(-1) if t.numel() > 1 else t
        t_vec = t_val * torch.ones((1, 1, 1), device=target_device, dtype=target_dtype)

        # 4. Динамика
        delta_h = self.time_embed(t_vec)
        h_t = h_in + delta_h

        # Проход через слой (Кэш отключен, так как в ODE нужна полная траектория)
        layer_outputs = self.base_layer(
            h_t,
            attention_mask=self.context.get("attention_mask"),
            position_ids=self.context.get("position_ids"),
            position_embeddings=self.context.get("position_embeddings"),
            use_cache=False,
        )

        output_h = (
            layer_outputs[0] if isinstance(layer_outputs, tuple) else layer_outputs
        )

        # 5. Производная: f(h) = Layer(h) - h (ResNet formulation -> ODE)
        derivative = output_h - h_t

        # 6. Возврат в Float32 для солвера
        return derivative.to(torch.float32)


# =============================================================================
# 2. ODE MODEL CORE
# =============================================================================


class Qwen3ModelODE(nn.Module):
    def __init__(self, original_model_internal):
        super().__init__()
        self.config = original_model_internal.config
        self.embed_tokens = original_model_internal.embed_tokens
        self.layers = original_model_internal.layers
        self.norm = original_model_internal.norm
        self.rotary_emb = original_model_internal.rotary_emb

        # Используем средний слой как функцию f(x)
        self.dynamics_layer_idx = len(self.layers) // 2

        # Определяем тип внимания
        target_layer = self.layers[self.dynamics_layer_idx]
        self.target_attn_type = getattr(
            target_layer, "attention_type", "full_attention"
        )

        self.ode_dynamics = None
        self.rtol = 1e-3
        self.atol = 1e-3

    def get_dynamics(self):
        if self.ode_dynamics is None:
            target_layer = self.layers[self.dynamics_layer_idx]
            # Важно: убеждаемся, что параметры на месте
            ref_param = target_layer.input_layernorm.weight

            self.ode_dynamics = ODEDynamicsWrapper(
                target_layer, self.config.hidden_size
            )
            self.ode_dynamics.to(device=ref_param.device, dtype=ref_param.dtype)
        return self.ode_dynamics

    def forward(
        self,
        input_ids: Optional[torch.LongTensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
        position_ids: Optional[torch.LongTensor] = None,
        past_key_values: Optional[List[torch.FloatTensor]] = None,  # Игнорируем в ODE
        inputs_embeds: Optional[torch.FloatTensor] = None,
        use_cache: Optional[bool] = None,
        cache_position: Optional[torch.LongTensor] = None,
        **kwargs,
    ):
        # 1. Эмбеддинги
        if inputs_embeds is None:
            inputs_embeds = self.embed_tokens(input_ids)

        # Neural ODE не поддерживает стандартный KV-Cache (past_key_values),
        # так как состояние h(t) меняется непрерывно. Мы всегда пересчитываем полный контекст.

        batch_size, seq_length, _ = inputs_embeds.shape
        device = inputs_embeds.device

        # 2. Позиции (упрощено для ODE, считаем всегда полный контекст)
        if position_ids is None:
            position_ids = torch.arange(seq_length, device=device).unsqueeze(0)

        # 3. Маска
        # Qwen2.5 использует FlashAttn, поэтому маска часто None, но создадим для надежности
        if attention_mask is None:
            # Простая каузальная маска
            attention_mask = torch.ones((batch_size, seq_length), device=device)

        # Для create_causal_mask нужны 4D маски в старых версиях, но современные модели сами справляются.
        # Для Qwen/FlashAttn передаем сырую маску в динамику, пусть слой сам разбирается.
        # Но чтобы ODEDynamicsWrapper работал корректно с forward слоя, подготовим standard mask если нужно.
        # (Здесь мы полагаемся на то, что layer() внутри ODEDynamicsWrapper сам вызовет _update_causal_mask если надо)

        # 4. RoPE
        # Qwen Rotary Embedding
        position_embeddings = self.rotary_emb(inputs_embeds, position_ids)

        # =======================================================
        # ODE INTEGRATION
        # =======================================================

        # Начальное состояние (Float32 для солвера)
        hidden_states_f32 = inputs_embeds.to(torch.float32)

        dynamics = self.get_dynamics()

        # Передаем контекст (маски, ропе) в обертку
        dynamics.set_context(
            attention_mask=attention_mask,  # Можно передать None для FlashAttn
            position_ids=position_ids,
            position_embeddings=position_embeddings,
        )

        # Интегрируем от 0 до 1
        t_span = torch.tensor([0.0, 1.0], device=device, dtype=torch.float32)

        integrated_states = odeint(
            dynamics,
            hidden_states_f32,
            t_span,
            rtol=self.rtol,
            atol=self.atol,
            method="dopri5",  # Runge-Kutta 4(5)
            # method="euler", # Можно переключить на euler для скорости (но ниже точность)
        )

        # Берем состояние в t=1 и возвращаем исходный тип данных
        final_states = integrated_states[-1].to(inputs_embeds.dtype)
        final_states = self.norm(final_states)

        return BaseModelOutputWithPast(
            last_hidden_state=final_states,
            past_key_values=None,  # Кэш не возвращаем
        )


# =============================================================================
# 3. ГЛАВНАЯ ОБЕРТКА (С ИСПРАВЛЕННОЙ ГЕНЕРАЦИЕЙ)
# =============================================================================


class Qwen3ForCausalLMODE(PreTrainedModel, GenerationMixin):
    _is_stateful = False

    # === ДОБАВЛЕННЫЕ ФЛАГИ ===
    # Сообщаем библиотеке, что наша модель поддерживает Flash Attention 2
    _supports_flash_attn_2 = True
    _supports_sdpa = True  # Scaled Dot Product Attention (PyTorch native)

    def __init__(self, original_causal_lm):
        super().__init__(original_causal_lm.config)
        self.model = Qwen3ModelODE(original_causal_lm.model)
        self.lm_head = original_causal_lm.lm_head

        # Копируем конфиг генерации, отключаем кэш для ODE
        self.generation_config = copy.deepcopy(original_causal_lm.generation_config)
        self.generation_config.use_cache = False

    def get_input_embeddings(self):
        return self.model.embed_tokens

    def set_input_embeddings(self, value):
        self.model.embed_tokens = value

    def forward(
        self,
        input_ids: Optional[torch.LongTensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
        position_ids: Optional[torch.LongTensor] = None,
        past_key_values: Optional[List[torch.FloatTensor]] = None,
        inputs_embeds: Optional[torch.FloatTensor] = None,
        labels: Optional[torch.LongTensor] = None,
        use_cache: Optional[bool] = None,
        **kwargs,
    ):
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            position_ids=position_ids,
            inputs_embeds=inputs_embeds,
            **kwargs,
        )

        hidden_states = outputs.last_hidden_state
        logits = self.lm_head(hidden_states)

        loss = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(
                shift_logits.view(-1, self.config.vocab_size), shift_labels.view(-1)
            )

        return CausalLMOutputWithPast(
            loss=loss,
            logits=logits,
            past_key_values=None,
            hidden_states=hidden_states,
        )

    def prepare_inputs_for_generation(
        self,
        input_ids,
        past_key_values=None,
        attention_mask=None,
        inputs_embeds=None,
        **kwargs,
    ):
        # Игнорируем past_key_values, требуем полный контекст для ODE
        if inputs_embeds is not None:
            return {
                "inputs_embeds": inputs_embeds,
                "attention_mask": attention_mask,
                "use_cache": False,
            }

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "position_ids": kwargs.get("position_ids"),
            "use_cache": False,
        }


# =============================================================================
# 4. ЗАПУСК
# =============================================================================

if __name__ == "__main__":
    print(">>> [1/4] Инициализация...")
    model_name = "Qwen/Qwen2.5-0.5B"

    dtype = (
        torch.bfloat16
        if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
        else torch.float16
    )
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if device == "cpu":
        dtype = torch.float32

    print(f"Загрузка оригинальной модели: {model_name} ({dtype})")

    # Загружаем исходную модель
    original_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=dtype,
        device_map=device,
        trust_remote_code=True,
        attn_implementation="flash_attention_2" if device == "cuda" else "eager",
    )

    print(">>> [2/4] Конвертация в Neural ODE...")
    # Создаем нашу ODE модель
    ode_model = Qwen3ForCausalLMODE(original_model)

    # Перемещаем на девайс (важно, т.к. мы пересобрали модель)
    ode_model.to(device)
    ode_model.eval()

    # Инициализируем динамику один раз перед запуском
    ode_model.model.get_dynamics()

    print(">>> [3/4] Тест forward pass...")
    input_text = "The logic of scientific discovery is"
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(device)

    with torch.no_grad():
        # Проверка обычного вызова
        outputs = ode_model(**inputs)
        next_token_logits = outputs.logits[:, -1, :]
        next_token = torch.argmax(next_token_logits, dim=-1)
        print(f"Вход: '{input_text}'")
        print(f"Прогноз (1 шаг): '{tokenizer.decode(next_token)}'")

    print("\n>>> [4/4] Генерация текста (ODE Solver)...")
    print("(Это будет медленнее обычного, так как кэш отключен и решается ОДУ)")

    # Настройка генерации
    gen_kwargs = {
        "max_new_tokens": 10,
        "do_sample": True,
        "temperature": 0.7,
        "top_k": 50,
        "pad_token_id": tokenizer.eos_token_id,
        "use_cache": False,  # Принудительно отключаем кэш снаружи
    }

    output_ids = ode_model.generate(inputs.input_ids, **gen_kwargs)

    generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    print("-" * 40)
    print("РЕЗУЛЬТАТ ODE:")
    print(generated_text)
    print("-" * 40)

Skipping import of cpp extensions due to incompatible torch version 2.8.0+cu129 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info


>>> [1/4] Инициализация...
Загрузка оригинальной модели: Qwen/Qwen2.5-0.5B (torch.bfloat16)


`torch_dtype` is deprecated! Use `dtype` instead!


>>> [2/4] Конвертация в Neural ODE...
>>> [3/4] Тест forward pass...


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Вход: 'The logic of scientific discovery is'
Прогноз (1 шаг): ' negó'

>>> [4/4] Генерация текста (ODE Solver)...
(Это будет медленнее обычного, так как кэш отключен и решается ОДУ)
----------------------------------------
РЕЗУЛЬТАТ ODE:
The logic of scientific discovery is negó negó negó negó negó negó negó negó negó negó
----------------------------------------


In [1]:
import torch
import torch.nn as nn
import copy
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    PreTrainedModel,
    GenerationMixin,
)
from transformers.modeling_outputs import (
    BaseModelOutputWithPast,
    CausalLMOutputWithPast,
)

try:
    from torchdiffeq import odeint_adjoint as odeint
except ImportError:
    raise ImportError("Установите библиотеку: pip install torchdiffeq")


# =============================================================================
# 1. ДИНАМИКА (STACKED ODE)
# =============================================================================


class StackedODEDynamics(nn.Module):
    def __init__(self, layers: nn.ModuleList, hidden_size: int):
        super().__init__()
        self.layers = layers
        self.time_embed_net = nn.Sequential(
            nn.Linear(1, hidden_size),
            nn.SiLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.Tanh(),
        )
        self.context = {}

    def set_context(self, **kwargs):
        self.context = kwargs

    def forward(self, t, h_state):
        # Приводим к типу весов модели
        ref_weight = self.layers[0].input_layernorm.weight
        target_dtype = ref_weight.dtype

        h_in = h_state.to(target_dtype)

        # Время t
        t_vec = t.view(-1).to(h_in.device, dtype=target_dtype)
        t_emb = self.time_embed_net(t_vec.view(1, 1, 1))

        x = h_in + t_emb

        # Проход через стек
        for layer in self.layers:
            # Важно: здесь мы передаем контекст (маски и rope)
            layer_out = layer(
                x,
                attention_mask=self.context.get("attention_mask"),
                position_ids=self.context.get("position_ids"),
                position_embeddings=self.context.get("position_embeddings"),
                use_cache=False,
            )
            x = layer_out[0]

        # Производная: dh/dt = f(h) - h
        derivative = x - h_in
        return derivative.to(torch.float32)


# =============================================================================
# 2. ГИБРИДНАЯ МОДЕЛЬ
# =============================================================================


class HybridQwenODE(nn.Module):
    def __init__(
        self, original_model, n_start_layers=2, n_ode_layers=2, n_end_layers=2
    ):
        super().__init__()
        self.config = original_model.config
        self.embed_tokens = original_model.embed_tokens
        self.norm = original_model.norm
        self.rotary_emb = original_model.rotary_emb

        all_layers = original_model.layers

        # 1. Начало
        self.pre_layers = nn.ModuleList([all_layers[i] for i in range(n_start_layers)])

        # 2. ODE Центр
        ode_idx_start = n_start_layers
        ode_idx_end = n_start_layers + n_ode_layers
        self.ode_dynamics = StackedODEDynamics(
            nn.ModuleList([all_layers[i] for i in range(ode_idx_start, ode_idx_end)]),
            self.config.hidden_size,
        )

        # 3. Конец
        final_idx_start = len(all_layers) - n_end_layers
        self.post_layers = nn.ModuleList(
            [all_layers[i] for i in range(final_idx_start, len(all_layers))]
        )

        self.rtol = 1e-3
        self.atol = 1e-3
        self.integration_time = 1.0

    def forward(
        self,
        input_ids,
        attention_mask=None,
        position_ids=None,
        inputs_embeds=None,
        **kwargs,
    ):
        if inputs_embeds is None:
            inputs_embeds = self.embed_tokens(input_ids)

        seq_len = inputs_embeds.shape[1]
        device = inputs_embeds.device

        if position_ids is None:
            position_ids = (
                torch.arange(seq_len, device=device)
                .unsqueeze(0)
                .expand(inputs_embeds.shape[0], -1)
            )

        # === КЛЮЧЕВОЕ ИСПРАВЛЕНИЕ: генерируем cos/sin правильного размера ===
        # rotary_emb ожидает inv_freq, но в HF он вызывается на query/key внутри слоя.
        # Мы делаем так же, как делает оригинальная модель — передаём None,
        # а внутри слоёв используется self.rotary_emb(dim=head_dim)
        # Поэтому здесь мы просто передаём position_ids, а cos/sin слой сам себе посчитает.
        # НО! В Qwen2 реализация позволяет вызвать rotary_emb с seq_len:
        cos, sin = self.rotary_emb(inputs_embeds, seq_len=seq_len)  # это работает!
        position_embeddings = (cos, sin)
        # ===============================================================

        h = inputs_embeds

        # Pre-layers
        for layer in self.pre_layers:
            h = layer(
                h,
                attention_mask=attention_mask,
                position_ids=position_ids,
                position_embeddings=position_embeddings,
                use_cache=False,
            )[0]

        # ODE block
        self.ode_dynamics.set_context(
            attention_mask=attention_mask,
            position_ids=position_ids,
            position_embeddings=position_embeddings,
        )

        h0 = h.to(torch.float32)
        t_span = torch.tensor(
            [0.0, self.integration_time], device=device, dtype=torch.float32
        )

        trajectory = odeint(
            self.ode_dynamics,
            h0,
            t_span,
            rtol=self.rtol,
            atol=self.atol,
            method="dopri5",  # или "rk4" если dopri5 слишком медленный
        )
        h = trajectory[-1].to(inputs_embeds.dtype)

        # Post-layers
        for layer in self.post_layers:
            h = layer(
                h,
                attention_mask=attention_mask,
                position_ids=position_ids,
                position_embeddings=position_embeddings,
                use_cache=False,
            )[0]

        h = self.norm(h)
        return BaseModelOutputWithPast(last_hidden_state=h)


# =============================================================================
# 3. ОБЕРТКА HF
# =============================================================================


class HybridODEForCausalLM(PreTrainedModel, GenerationMixin):
    _is_stateful = False

    # Флаги поддержки SDPA и FlashAttn
    _supports_flash_attn_2 = True
    _supports_sdpa = True

    def __init__(self, original_causal_lm, **kwargs):
        super().__init__(original_causal_lm.config)
        self.model = HybridQwenODE(original_causal_lm.model, **kwargs)
        self.lm_head = original_causal_lm.lm_head

        self.generation_config = copy.deepcopy(original_causal_lm.generation_config)
        self.generation_config.use_cache = False

    def forward(self, input_ids=None, labels=None, attention_mask=None, **kwargs):
        outputs = self.model(
            input_ids=input_ids, attention_mask=attention_mask, **kwargs
        )
        hidden_states = outputs.last_hidden_state
        logits = self.lm_head(hidden_states)

        loss = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(
                shift_logits.view(-1, self.config.vocab_size), shift_labels.view(-1)
            )

        return CausalLMOutputWithPast(
            loss=loss, logits=logits, hidden_states=hidden_states
        )

    def prepare_inputs_for_generation(self, input_ids, **kwargs):
        return {
            "input_ids": input_ids,
            "attention_mask": kwargs.get("attention_mask", None),
            "use_cache": False,
        }


# =============================================================================
# 4. ЗАПУСК
# =============================================================================

if __name__ == "__main__":
    print(">>> Загрузка модели...")
    model_id = "Qwen/Qwen2.5-0.5B"

    orig_model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16,
        device_map="cuda",
        attn_implementation="sdpa",
    )
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    # Инициализация Neural ODE
    ode_model = HybridODEForCausalLM(
        orig_model, n_start_layers=2, n_ode_layers=2, n_end_layers=2
    )
    ode_model.to("cuda")
    ode_model.train()

    optimizer = torch.optim.AdamW(ode_model.parameters(), lr=1e-4)

    text = "Neural ODEs represent a continuous-depth deep learning framework."
    inputs = tokenizer(text, return_tensors="pt").to("cuda")

    print(f"\n>>> Тест обучения на фразе: '{text}'")

    for step in range(21):
        outputs = ode_model(input_ids=inputs.input_ids, labels=inputs.input_ids)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()  # Здесь работает Adjoint Method
        optimizer.step()

        if step % 5 == 0:
            print(f"Step {step:02d} | Loss: {loss.item():.5f}")

    print("\n>>> Тест генерации:")
    ode_model.eval()
    prompt = "Neural ODEs represent"
    gen_ids = ode_model.generate(
        **tokenizer(prompt, return_tensors="pt").to("cuda"), max_new_tokens=10
    )
    print("Результат:", tokenizer.decode(gen_ids[0], skip_special_tokens=True))

Skipping import of cpp extensions due to incompatible torch version 2.8.0+cu129 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info


>>> Загрузка модели...


`torch_dtype` is deprecated! Use `dtype` instead!



>>> Тест обучения на фразе: 'Neural ODEs represent a continuous-depth deep learning framework.'


TypeError: Qwen2RotaryEmbedding.forward() got an unexpected keyword argument 'seq_len'

In [ ]:
import torch
import torch.nn as nn
import copy
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    PreTrainedModel,
    GenerationMixin,
)
from transformers.modeling_outputs import (
    BaseModelOutputWithPast,
    CausalLMOutputWithPast,
)

import torch
import torch.nn as nn
import copy
from typing import Optional, Tuple, Union, List
from transformers.masking_utils import create_causal_mask

try:
    from torchdiffeq import odeint_adjoint as odeint
except ImportError:
    raise ImportError("Установите библиотеку: pip install torchdiffeq")


# =============================================================================
# 1. ДИНАМИКА (STACKED ODE)
# =============================================================================


class StackedODEDynamics(nn.Module):
    def __init__(self, layers: nn.ModuleList, hidden_size: int):
        super().__init__()
        self.layers = layers
        self.time_embed_net = nn.Sequential(
            nn.Linear(1, hidden_size),
            nn.SiLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.Tanh(),
        )
        self.context = {}

    def set_context(self, **kwargs):
        self.context = kwargs

    def forward(self, t, h_state):
        ref_weight = self.layers[0].input_layernorm.weight
        target_dtype = ref_weight.dtype
        h_in = h_state.to(target_dtype)

        t_vec = t.view(-1).to(h_in.device, dtype=target_dtype)
        t_emb = self.time_embed_net(t_vec.view(1, 1, 1))
        x = h_in + t_emb

        for layer in self.layers:
            # ИСПРАВЛЕНИЕ: Передаем пред-вычисленные position_embeddings из контекста
            layer_out = layer(
                x,
                attention_mask=self.context.get("attention_mask"),
                position_ids=self.context.get("position_ids"),
                position_embeddings=self.context.get("position_embeddings"),
                use_cache=False,
            )
            x = layer_out[0]

        derivative = x - h_in
        return derivative.to(torch.float32)


# =============================================================================
# 2. ГИБРИДНАЯ МОДЕЛЬ
# =============================================================================

from transformers.cache_utils import Cache, DynamicCache


class HybridQwenODE(nn.Module):
    def __init__(
        self, original_model, n_start_layers=2, n_ode_layers=2, n_end_layers=2
    ):
        super().__init__()
        self.config = original_model.config
        self.embed_tokens = original_model.embed_tokens
        self.norm = original_model.norm
        # ВОЗВРАЩАЕМ: rotary_emb нужен для пред-вычисления
        self.rotary_emb = original_model.rotary_emb

        all_layers = original_model.layers
        self.pre_layers = nn.ModuleList([all_layers[i] for i in range(n_start_layers)])

        ode_idx_start = n_start_layers
        ode_idx_end = n_start_layers + n_ode_layers
        self.ode_dynamics = StackedODEDynamics(
            nn.ModuleList([all_layers[i] for i in range(ode_idx_start, ode_idx_end)]),
            self.config.hidden_size,
        )

        final_idx_start = len(all_layers) - n_end_layers
        self.post_layers = nn.ModuleList(
            [all_layers[i] for i in range(final_idx_start, len(all_layers))]
        )

        self.rtol = 1e-3
        self.atol = 1e-3
        self.integration_time = 1.0

    def forward(
        self,
        input_ids: Optional[torch.LongTensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
        position_ids: Optional[torch.LongTensor] = None,
        past_key_values=None,
        inputs_embeds: Optional[torch.FloatTensor] = None,
        use_cache: Optional[bool] = None,
        cache_position: Optional[torch.LongTensor] = None,
        **kwargs,
    ) -> BaseModelOutputWithPast:

        if (input_ids is None) ^ (inputs_embeds is not None):
            raise ValueError(
                "You must specify exactly one of input_ids or inputs_embeds"
            )

        if inputs_embeds is None:
            inputs_embeds = self.embed_tokens(input_ids)

        if use_cache and past_key_values is None:
            past_key_values = DynamicCache(config=self.config)

        if cache_position is None:
            past_seen_tokens = (
                past_key_values.get_seq_length() if past_key_values is not None else 0
            )
            cache_position = torch.arange(
                past_seen_tokens,
                past_seen_tokens + inputs_embeds.shape[1],
                device=inputs_embeds.device,
            )

        if position_ids is None:
            position_ids = cache_position.unsqueeze(0)

        # It may already have been prepared by e.g. `generate`
        if not isinstance(causal_mask_mapping := attention_mask, dict):
            # Prepare mask arguments
            mask_kwargs = {
                "config": self.config,
                "input_embeds": inputs_embeds,
                "attention_mask": attention_mask,
                "cache_position": cache_position,
                "past_key_values": past_key_values,
                "position_ids": position_ids,
            }
            # Create the masks
            causal_mask_mapping = {
                "full_attention": create_causal_mask(**mask_kwargs),
            }
            # The sliding window alternating layers are not always activated depending on the config
            # if self.has_sliding_layers:
            #     causal_mask_mapping["sliding_attention"] = (
            #         create_sliding_window_causal_mask(**mask_kwargs)
            #     )

        hidden_states = inputs_embeds

        # create position embeddings to be shared across the decoder layers
        position_embeddings = self.rotary_emb(hidden_states, position_ids)

        for decoder_layer in self.pre_layers[: self.config.num_hidden_layers]:
            hidden_states = decoder_layer(
                hidden_states,
                attention_mask=causal_mask_mapping[decoder_layer.attention_type],
                position_ids=position_ids,
                past_key_values=past_key_values,
                use_cache=use_cache,
                cache_position=cache_position,
                position_embeddings=position_embeddings,
                **kwargs,
            )
        h = hidden_states

        # Phase 2: ODE Solver
        self.ode_dynamics.set_context(
            attention_mask=attention_mask,
            position_ids=position_ids,
            position_embeddings=position_embeddings,
        )

        h0 = h.to(torch.float32)
        t_span = torch.tensor([0.0, self.integration_time], device=inputs_embeds.device)

        trajectory = odeint(
            self.ode_dynamics,
            h0,
            t_span,
            rtol=self.rtol,
            atol=self.atol,
            method="dopri5",
        )
        h = trajectory[-1].to(inputs_embeds.dtype)

        # Phase 3: Post-layers
        for layer in self.post_layers:
            h = layer(
                h,
                attention_mask=attention_mask,
                position_ids=position_ids,
                position_embeddings=position_embeddings,
                use_cache=False,
            )[0]

        h = self.norm(h)
        # h = self.norm(hidden_states)
        return BaseModelOutputWithPast(last_hidden_state=h)


# =============================================================================
# 3. ОБЕРТКА HF (Без изменений)
# =============================================================================


class HybridODEForCausalLM(PreTrainedModel, GenerationMixin):
    _is_stateful = False
    _supports_flash_attn_2 = True
    _supports_sdpa = True

    def __init__(self, original_causal_lm, **kwargs):
        super().__init__(original_causal_lm.config)
        self.model = HybridQwenODE(original_causal_lm.model, **kwargs)
        self.lm_head = original_causal_lm.lm_head
        self.generation_config = copy.deepcopy(original_causal_lm.generation_config)
        self.generation_config.use_cache = False

    def forward(self, input_ids=None, labels=None, attention_mask=None, **kwargs):
        outputs = self.model(
            input_ids=input_ids, attention_mask=attention_mask, **kwargs
        )
        hidden_states = outputs.last_hidden_state
        logits = self.lm_head(hidden_states)

        loss = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(
                shift_logits.view(-1, self.config.vocab_size), shift_labels.view(-1)
            )

        return CausalLMOutputWithPast(
            loss=loss, logits=logits, hidden_states=hidden_states
        )

    def prepare_inputs_for_generation(self, input_ids, **kwargs):
        # Важно передавать position_ids для корректной работы RoPE при генерации
        # В данном простом случае это неявно обработается в forward, но для KV-кэша это было бы критично
        return {
            "input_ids": input_ids,
            "attention_mask": kwargs.get("attention_mask", None),
            "position_ids": kwargs.get("position_ids", None),
            "use_cache": False,
        }


# =============================================================================
# 4. ЗАПУСК (Без изменений)
# =============================================================================

if __name__ == "__main__":
    print(">>> Загрузка модели...")
    model_id = "Qwen/Qwen2.5-0.5B"

    orig_model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16,
        device_map="cuda",
        attn_implementation="sdpa",
    )
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    ode_model = HybridODEForCausalLM(
        orig_model, n_start_layers=2, n_ode_layers=2, n_end_layers=2
    )
    ode_model.to("cuda")
    ode_model.train()

    optimizer = torch.optim.AdamW(ode_model.parameters(), lr=1e-4)

    text = "Neural ODEs represent a continuous-depth deep learning framework."
    inputs = tokenizer(text, return_tensors="pt").to("cuda")

    print(f"\n>>> Тест обучения на фразе: '{text}'")

    for step in range(21):
        outputs = ode_model(input_ids=inputs.input_ids, labels=inputs.input_ids)
        loss = outputs.loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if step % 5 == 0:
            print(f"Step {step:02d} | Loss: {loss.item():.5f}")

    print("\n>>> Тест генерации:")
    ode_model.eval()
    prompt = "Neural ODEs represent"
    gen_ids = ode_model.generate(
        **tokenizer(prompt, return_tensors="pt").to("cuda"), max_new_tokens=10
    )
    print("Результат:", tokenizer.decode(gen_ids[0], skip_special_tokens=True))

>>> Загрузка модели...

>>> Тест обучения на фразе: 'Neural ODEs represent a continuous-depth deep learning framework.'


RuntimeError: mat1 and mat2 must have the same dtype, but got BFloat16 and Float

In [1]:
import torch
import torch.nn as nn
import copy
from typing import Optional, Tuple, Union, List
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    PreTrainedModel,
    GenerationMixin,
)
from transformers.modeling_outputs import (
    BaseModelOutputWithPast,
    CausalLMOutputWithPast,
)
from transformers.cache_utils import Cache, DynamicCache
from transformers.masking_utils import create_causal_mask

try:
    from torchdiffeq import odeint_adjoint as odeint
except ImportError:
    raise ImportError("Установите библиотеку: pip install torchdiffeq")


# =============================================================================
# 1. ДИНАМИКА (STACKED ODE)
# =============================================================================


class StackedODEDynamics(nn.Module):
    def __init__(self, layers: nn.ModuleList, hidden_size: int):
        super().__init__()
        self.layers = layers
        self.time_embed_net = nn.Sequential(
            nn.Linear(1, hidden_size),
            nn.SiLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.Tanh(),
        )
        self.context = {}

    def set_context(self, **kwargs):
        self.context = kwargs

    def forward(self, t, h_state):
        ref_weight = self.layers[0].input_layernorm.weight
        target_dtype = ref_weight.dtype

        h_in = h_state.to(target_dtype)
        t_vec = t.view(-1).to(h_in.device, dtype=target_dtype)

        t_emb = self.time_embed_net(t_vec.view(1, 1, 1))
        x = h_in + t_emb

        for layer in self.layers:
            # ИСПРАВЛЕНИЕ: Убрали position_embeddings. Слой сам вычислит их по position_ids.
            layer_out = layer(
                x,
                attention_mask=self.context.get("attention_mask"),
                position_ids=self.context.get("position_ids"),
                use_cache=False,
            )
            x = layer_out[0]

        derivative = x - h_in
        return derivative.to(torch.float32)


# =============================================================================
# 2. ГИБРИДНАЯ МОДЕЛЬ
# =============================================================================


class HybridQwenODE(nn.Module):
    def __init__(
        self, original_model, n_start_layers=2, n_ode_layers=2, n_end_layers=2
    ):
        super().__init__()
        self.config = original_model.config
        self.embed_tokens = original_model.embed_tokens
        self.norm = original_model.norm
        # rotary_emb больше не нужен здесь явно, но оставляем для совместимости
        self.rotary_emb = original_model.rotary_emb

        all_layers = original_model.layers
        self.pre_layers = nn.ModuleList([all_layers[i] for i in range(n_start_layers)])

        ode_idx_start = n_start_layers
        ode_idx_end = n_start_layers + n_ode_layers

        self.ode_dynamics = StackedODEDynamics(
            nn.ModuleList([all_layers[i] for i in range(ode_idx_start, ode_idx_end)]),
            self.config.hidden_size,
        )

        # Исправление dtype для time_embed_net
        ref_dtype = self.pre_layers[0].input_layernorm.weight.dtype
        self.ode_dynamics.time_embed_net.to(dtype=ref_dtype)

        final_idx_start = len(all_layers) - n_end_layers
        self.post_layers = nn.ModuleList(
            [all_layers[i] for i in range(final_idx_start, len(all_layers))]
        )

        self.rtol = 1e-3
        self.atol = 1e-3
        self.integration_time = 1.0

    def forward(
        self,
        input_ids: Optional[torch.LongTensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
        position_ids: Optional[torch.LongTensor] = None,
        past_key_values=None,
        inputs_embeds: Optional[torch.FloatTensor] = None,
        use_cache: Optional[bool] = None,
        cache_position: Optional[torch.LongTensor] = None,
        **kwargs,
    ) -> BaseModelOutputWithPast:

        if (input_ids is None) ^ (inputs_embeds is not None):
            raise ValueError(
                "You must specify exactly one of input_ids or inputs_embeds"
            )

        if inputs_embeds is None:
            inputs_embeds = self.embed_tokens(input_ids)

        if use_cache and past_key_values is None:
            past_key_values = DynamicCache(config=self.config)

        if cache_position is None:
            past_seen_tokens = (
                past_key_values.get_seq_length() if past_key_values is not None else 0
            )
            cache_position = torch.arange(
                past_seen_tokens,
                past_seen_tokens + inputs_embeds.shape[1],
                device=inputs_embeds.device,
            )

        if position_ids is None:
            position_ids = cache_position.unsqueeze(0)

        if not isinstance(causal_mask_mapping := attention_mask, dict):
            mask_kwargs = {
                "config": self.config,
                "input_embeds": inputs_embeds,
                "attention_mask": attention_mask,
                "cache_position": cache_position,
                "past_key_values": past_key_values,
                "position_ids": position_ids,
            }
            causal_mask_mapping = {
                "full_attention": create_causal_mask(**mask_kwargs),
            }

        hidden_states = inputs_embeds

        # ИСПРАВЛЕНИЕ: Не вычисляем position_embeddings вручную.
        # Слои сами это сделают корректно.

        # Phase 1: Pre-layers
        for decoder_layer in self.pre_layers[: self.config.num_hidden_layers]:
            hidden_states = decoder_layer(
                hidden_states,
                attention_mask=causal_mask_mapping[decoder_layer.attention_type],
                position_ids=position_ids,
                past_key_values=past_key_values,
                use_cache=use_cache,
                cache_position=cache_position,
                # position_embeddings=... удалено
                **kwargs,
            )
        h = hidden_states

        # Phase 2: ODE Solver
        self.ode_dynamics.set_context(
            attention_mask=attention_mask,
            position_ids=position_ids,
            # position_embeddings удалено из контекста
        )

        h0 = h.to(torch.float32)
        t_span = torch.tensor([0.0, self.integration_time], device=inputs_embeds.device)

        trajectory = odeint(
            self.ode_dynamics,
            h0,
            t_span,
            rtol=self.rtol,
            atol=self.atol,
            method="dopri5",
        )
        h = trajectory[-1].to(inputs_embeds.dtype)

        # Phase 3: Post-layers
        for layer in self.post_layers:
            h = layer(
                h,
                attention_mask=attention_mask,
                position_ids=position_ids,
                use_cache=False,
            )[0]

        h = self.norm(h)
        return BaseModelOutputWithPast(last_hidden_state=h)


# =============================================================================
# 3. ОБЕРТКА HF
# =============================================================================


class HybridODEForCausalLM(PreTrainedModel, GenerationMixin):
    _is_stateful = False
    _supports_flash_attn_2 = True
    _supports_sdpa = True

    def __init__(self, original_causal_lm, **kwargs):
        super().__init__(original_causal_lm.config)
        self.model = HybridQwenODE(original_causal_lm.model, **kwargs)
        self.lm_head = original_causal_lm.lm_head
        self.generation_config = copy.deepcopy(original_causal_lm.generation_config)
        self.generation_config.use_cache = False

    def forward(self, input_ids=None, labels=None, attention_mask=None, **kwargs):
        outputs = self.model(
            input_ids=input_ids, attention_mask=attention_mask, **kwargs
        )
        hidden_states = outputs.last_hidden_state
        logits = self.lm_head(hidden_states)

        loss = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(
                shift_logits.view(-1, self.config.vocab_size), shift_labels.view(-1)
            )

        return CausalLMOutputWithPast(
            loss=loss, logits=logits, hidden_states=hidden_states
        )

    def prepare_inputs_for_generation(self, input_ids, **kwargs):
        return {
            "input_ids": input_ids,
            "attention_mask": kwargs.get("attention_mask", None),
            "position_ids": kwargs.get("position_ids", None),
            "use_cache": False,
        }


# =============================================================================
# 4. ЗАПУСК
# =============================================================================

if __name__ == "__main__":
    print(">>> Загрузка модели...")
    model_id = "Qwen/Qwen2.5-0.5B"

    orig_model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16,
        device_map="cuda",
        attn_implementation="sdpa",
    )
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    ode_model = HybridODEForCausalLM(
        orig_model, n_start_layers=2, n_ode_layers=2, n_end_layers=2
    )

    # Явно приводим к типу для надежности
    ode_model.to(device="cuda", dtype=torch.bfloat16)
    ode_model.train()

    optimizer = torch.optim.AdamW(ode_model.parameters(), lr=1e-4)

    text = "Neural ODEs represent a continuous-depth deep learning framework."
    inputs = tokenizer(text, return_tensors="pt").to("cuda")

    print(f"\n>>> Тест обучения на фразе: '{text}'")

    for step in range(6):
        outputs = ode_model(input_ids=inputs.input_ids, labels=inputs.input_ids)
        loss = outputs.loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        print(f"Step {step:02d} | Loss: {loss.item():.5f}")

    print("\n>>> Тест генерации:")
    ode_model.eval()
    prompt = "Neural ODEs represent"
    gen_ids = ode_model.generate(
        **tokenizer(prompt, return_tensors="pt").to("cuda"), max_new_tokens=10
    )
    print("Результат:", tokenizer.decode(gen_ids[0], skip_special_tokens=True))

Skipping import of cpp extensions due to incompatible torch version 2.8.0+cu129 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info


>>> Загрузка модели...


`torch_dtype` is deprecated! Use `dtype` instead!



>>> Тест обучения на фразе: 'Neural ODEs represent a continuous-depth deep learning framework.'


TypeError: cannot unpack non-iterable NoneType object